## Step 1) Download activity files (.gpx, .tcx)
i) save your Garmin login credentials by opening terminal/command prompt:  
```
export EMAIL={enter your garmin username/email}  
export PASSWORD={enter your garmin password}
```  
ii) set number of days from today to download in the following cell, then run the cell below to execute garmin_api.py

In [2]:
num_days_before_today = 14
## CHANGE ABOVE

#### executable: do not change below 
## navigate into project directory (location of script) & run from there  
## garmin_api.py saves activity files into a new folder named today's date (YYYYMMDD) located in the script/project directory   
out_dir = !python garmin_api.py $num_days_before_today
out_dir=out_dir[0]
out_dir

'/Users/orca/Documents/code/repos/geo-tlbx/running/20240617'

## Step 2) Parse new activity files, add to archive:     
* add parsed activity data into archive postgreSQL database, and move files to a single archive folder

In [ ]:
import os
import sys
sys.path.append("..")
import geo_utils as geou

## out_dir = output directory = folder with activity files 
## out_dir = '/Users/orca/Documents/code/repos/geo-tlbx/running/20240617'

################################################################################

## parse gpx and tcx files 
runGPX_df, new_pt_csv = geou.parse_gpx(out_dir)
runTCX_df, bikeTCX_df = geou.parse_tcx(out_dir)

## move each activity file to archive folder 
for option in file_types:
    move_files = [i for i in os.listdir(out_dir) if (i.endswith(option) and "X_" not in i)]
    for file in move_files:
        os.rename(os.path.join(out_dir, file), os.path.join(archive_dir, file))

## add entries to a postgreSQL database containing all activity files, run_db  
geou.update_gcx_table(df=runGPX_df, db="run_db", usr="postgres", pwd=os.getenv("POSTPWD"), localhost="localhost", port="5432")
geou.update_tcx_table(df=runTCX_df.reset_index(), db="run_db", usr="postgres", pwd=os.getenv("POSTPWD"), localhost="localhost", port="5432")

In [ ]:
## select activities from tcx_runs or points from gpx_runs 
SQL_query = "SELECT lon, lat, time, ele, speed FROM gpx_runs WHERE lat < 37.0"

items=geou.query_postgres(SQL_query=SQL_query, db="run_db", usr="postgres", pwd=os.getenv("POSTPWD"), localhost="localhost", port="5432")
items

## Step 3) Create plots & webmaps

In [ ]:
## route heatmap
plot_heatmap(Gdf, os.path.join(running_fig_dir, "heatmap.html"))    


In [ ]:
## 3d routes (set smaller bounds)
min_lon = -119.98
max_lon = -119.0
min_lat = 34.3
max_lat = 34.5

############################

df = Gdf[(Gdf['lat'] > min_lat) & (Gdf['lat'] < max_lat) & (Gdf['lon'] > min_lon) & (Gdf['lon'] < max_lon)]
plot_3d(df, os.path.join(running_fig_dir, "route3d.html"))    


In [ ]:
## calendar heatmap
cal_heatmap(Tdf, "miles")
cal_heatmap(Tdf, "ascent_m")


In [ ]:
!jupyter nbconvert running.ipynb --post serve --template classic --to html

[NbConvertApp] Converting notebook running.ipynb to html
[NbConvertApp] Writing 288864 bytes to running.html
[NbConvertApp] Redirecting reveal.js requests to https://cdnjs.cloudflare.com/ajax/libs/reveal.js/3.5.0
Serving your slides at http://127.0.0.1:8000/running.html
Use Control-C to stop this server
